# Data cutting
Rectilinear data cutting using variables across a ROOT file. Run each block once to initialize the cutting program

## Initialization
Run this block once to import all required libraries and initialize classes.

In [ ]:
import uproot
import csv
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from plotly.subplots import make_subplots
from ipyfilechooser import FileChooser
import plotly.graph_objects as go

# TODO: consider adding support for saving/loading a csv of cutter vars + values
    # notes + filename + vars (values)

class Events:
    def __init__(self, root_file_path):
        self.root_file = uproot.open(root_file_path.selected)
        self.root_file_name = root_file_path.selected_filename
        self.event_tree = self.root_file[self.root_file.keys()[1]]
        self._base_var = self.event_tree[0].name
        self._cutter_vars = {}
        self._original_array = []
        self._array_cache = {}

    @property
    def cutter_vars(self):
        return self._cutter_vars

    @cutter_vars.setter
    def cutter_vars(self, vars=[]):
        '''Takes in a list of cutter variables (strings) and appends them to variables.
        Pairs them each with a default range that covers 100% of their values.'''
        if isinstance(vars, list):
            for var in vars:
                if self.is_in_tree(var):
                    self.update_dict(var)
        else:
            if self.is_in_tree(vars):
                self.update_dict(vars)

    @property
    def base_var(self):
        return self._base_var

    @base_var.setter
    def base_var(self, var):
        self._base_var = var

    @property
    def original_array(self):
        '''Returns the unmodified numpy data array associated with the base variable'''
        return self.get_array(self._base_var)

    def get_array(self, var: str):
        '''Returns the data for a specified variable as a numpy array.'''
        if var not in self._array_cache:
            self._array_cache[var] = self.event_tree[var].array(library="np")
        return self._array_cache[var]

    def get_max_range(self, var: str):
        '''Returns the absolute (min, max) of a variable array.'''
        arr = self.get_array(var)
        return (arr.min(), arr.max())

    def get_display_range(self, var: str, lower_pct: float = 0.1, upper_pct: float = 99.9):
        '''Percentile-clipped range for histogram display; avoids outliers stretching the axis.'''
        arr = self.get_array(var)
        return (float(np.percentile(arr, lower_pct)), float(np.percentile(arr, upper_pct)))

    def set_var_range(self, var: str, range: tuple):
        '''Sets the cut range for a cutter variable.'''
        if var not in self._cutter_vars:
            raise Exception(f"Error: Cutter variable {var} is not in list of variables")
        self._cutter_vars[var] = range

    def get_total_mask(self):
        '''Returns a boolean mask based on all stored cutter ranges.'''
        if not self._cutter_vars:
            return np.ones(len(self.get_array(self._base_var)), dtype=bool)

        mask = np.isfinite(self.get_array(next(iter(self._cutter_vars))))
        for var in self._cutter_vars:
            arr = self.get_array(var)
            min_val = self._cutter_vars[var][0]
            max_val = self._cutter_vars[var][1]
            
            mask &= ((min_val <= arr) & (arr <= max_val))

        return mask

    def determine_default_vars(self, particle_vars: dict):
        '''Determines which list of default cutter variables to use by reading the decay tree title. Takes in a dict of particle names tied to lists of variables names.'''
        tree_title = self.root_file.keys()[0].lower()
        for particle in particle_vars:
            particle = particle.lower()
            if particle in tree_title:
                print(f"{self.root_file_name} contains tree {tree_title} of type {particle}")
                return particle_vars[particle]

    def is_bool_variable(self, var: str):
        '''Checks whether a variable contains boolean (0/1) data by sampling the first 100 values.'''
        for x in self.get_array(var)[:100]:
            if not (x == 1 or x == 0):
                return False
        return True

    def is_in_tree(self, var):
        '''For error handling; checks if a variable exists in the tree.'''
        tree = self.event_tree.keys()
        if var in tree:
            return True
        else:
            raise NameError(f"Variable {var} not found in decay tree!")

    def update_dict(self, var: str):
        '''Updates the object's dict of selected cutter variables and their ranges.'''
        if var not in self._cutter_vars.keys():
            full_range = self.get_max_range(var)
            self._cutter_vars.update({var: full_range})

    def export_vars(self, file="export/variable_export.csv"):
        csvfile = open(file, "w", newline="", encoding="utf-8")
        c = csv.writer(csvfile)
        
        # Write header and filename row
        c.writerow(["File", "Variable", "Min", "Max"])
        c.writerow([self.root_file_name, "", "", ""])

        # Write variable values
        for var in self._cutter_vars:
            min = self._cutter_vars[var][0]
            max = self._cutter_vars[var][1]

            c.writerow(["", var, min, max])
        
        csvfile.close()

class UIHelper():
    '''Object used to handle all UI elements and functions. Always tied to an Events object.'''
    def __init__(self, events_object):
        self.events = events_object

    def display_ui(self):
        '''Draws the UI and takes inputs. This is really ugly but ipywidgets freaks out if all of its functions aren't all grouped together like this.'''
        events = self.events

        # UI controls
        display_variable_dropdown = widgets.Dropdown(
            options=list(self.events.event_tree.keys()),
            description="Display Var:",
            value="B0_MM"
        )

        cutter_variable_dropdown = widgets.Dropdown(
            options=list(self.events.event_tree.keys()),
            description="Add var:",
        )

        bins_slider = widgets.IntSlider(
            value=300,
            min=100,
            max=5000,
            description="bins",
        )

        export_button = widgets.Button(
            description="Export vars",
            tooltip="Save to CSV in /export"
        )

        # Plotly figures (created once, updated in-place)
        base_fig = go.FigureWidget(
            make_subplots(rows=2, cols=1, shared_xaxes=True,
                          subplot_titles=["Original", "After cuts"])
        )
        base_fig.add_trace(go.Bar(name="Original", marker_color="steelblue", marker_line_width=0), row=1, col=1)           # Original base plot
        base_fig.add_trace(go.Bar(name="After cuts", marker_color="steelblue", marker_line_width=0), row=2, col=1)         # Base plot after cuts
        base_fig.update_layout(showlegend=False, height=500, bargap=0, yaxis_title="Number of events")

        cutter_fig = go.FigureWidget()
        cutter_fig.add_trace(go.Bar(name="Data", marker_color="steelblue", marker_line_width=0))            # Selected cutter variable plot
        cutter_fig.update_layout(title="Data Selection", showlegend=False, height=400, bargap=0)

        # Plot update functions
        def draw_base_plot():
            var = display_variable_dropdown.value
            bins = bins_slider.value
            original_arr = events.get_array(var)
            mask = events.get_total_mask()
            cut_arr = original_arr[mask]
            x_range = events.get_display_range(var)

            orig_counts, bin_edges = np.histogram(original_arr, bins=bins, range=x_range)
            cut_counts, _ = np.histogram(cut_arr, bins=bins, range=x_range)
            bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

            count_original = len(original_arr)
            count_cut = len(cut_arr)

            with base_fig.batch_update():
                base_fig.data[0].x = bin_centers
                base_fig.data[0].y = orig_counts
                base_fig.data[1].x = bin_centers
                base_fig.data[1].y = cut_counts
                base_fig.layout.xaxis.title.text = f"{var} ({count_original} events / 100%)"
                base_fig.layout.xaxis2.title.text = f"{var} ({count_cut} events / {round((count_cut/count_original)*100, 2)}%)"

        def draw_cutter_plot(var: str):
            range_vals = events.cutter_vars[var]

            if events.is_bool_variable(var):
                arr = events.get_array(var)
                count_false = int(np.sum(arr == 0))
                count_true  = int(np.sum(arr == 1))
                include_false = range_vals[0] <= 0 <= range_vals[1]
                include_true  = range_vals[0] <= 1 <= range_vals[1]

                with cutter_fig.batch_update():
                    cutter_fig.data[0].x = ["False (0)", "True (1)"]
                    cutter_fig.data[0].y = [count_false, count_true]
                    cutter_fig.data[0].marker.color = [
                        "steelblue" if include_false else "rgba(128,128,128,0.4)",
                        "steelblue" if include_true  else "rgba(128,128,128,0.4)",
                    ]
                    cutter_fig.layout.title.text       = f"Data Selection: {var} (boolean)"
                    cutter_fig.layout.xaxis.title.text = var
                    cutter_fig.layout.shapes = []
            else:
                display_range = events.get_display_range(var)
                counts, bin_edges = np.histogram(events.get_array(var), bins=500, range=display_range)
                bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

                with cutter_fig.batch_update():
                    cutter_fig.data[0].x = bin_centers
                    cutter_fig.data[0].y = counts
                    cutter_fig.data[0].marker.color = "steelblue"
                    cutter_fig.layout.title.text       = f"Data Selection: {var}"
                    cutter_fig.layout.xaxis.title.text = var
                    cutter_fig.layout.xaxis.range      = list(display_range)
                    cutter_fig.layout.update(shapes=[
                        dict(type="rect", x0=display_range[0], x1=range_vals[0],
                             y0=0, y1=1, xref="x", yref="paper",
                             fillcolor="grey", opacity=0.3, line_width=0, layer="below"),
                        dict(type="rect", x0=range_vals[1], x1=display_range[1],
                             y0=0, y1=1, xref="x", yref="paper",
                             fillcolor="grey", opacity=0.3, line_width=0, layer="below"),
                        dict(type="line", x0=range_vals[0], x1=range_vals[0],
                             y0=0, y1=1, xref="x", yref="paper",
                             line=dict(color="red", dash="dash")),
                        dict(type="line", x0=range_vals[1], x1=range_vals[1],
                             y0=0, y1=1, xref="x", yref="paper",
                             line=dict(color="red", dash="dash")),
                    ])

        # Widget observers
        def attach_slider_observer(slider, var):
            def on_slider_change(change):
                events.set_var_range(var, change["new"])
                draw_cutter_plot(var)
                draw_base_plot()
            slider.observe(on_slider_change, names="value")

        def attach_bool_observer(radio, var):
            def on_radio_change(change):
                sel = change["new"]
                if sel == "Both":
                    events.set_var_range(var, (0.0, 1.0))
                elif sel == "False only":
                    events.set_var_range(var, (0.0, 0.0))
                else:
                    events.set_var_range(var, (1.0, 1.0))
                draw_cutter_plot(var)
                draw_base_plot()
            radio.observe(on_radio_change, names="value")

        def make_cutter_widget(var):
            '''Factory: FloatRangeSlider for continuous vars, RadioButtons for booleans.
            Also registers the variable in events.cutter_vars if not already present.'''
            events.cutter_vars = var  # no-op if already registered

            if events.is_bool_variable(var):
                widget = widgets.RadioButtons(
                    options=["Both", "False only", "True only"],
                    value="Both",
                    description=var,
                    layout=widgets.Layout(width="auto"),
                )
                attach_bool_observer(widget, var)
            else:
                slider_range = events.get_display_range(var)
                widget = widgets.FloatRangeSlider(
                    min=slider_range[0],
                    max=slider_range[1],
                    step=(slider_range[1]-slider_range[0])/1000,
                    description=var,
                    value=slider_range,
                    layout=widgets.Layout(width="100%")
                )
                attach_slider_observer(widget, var)
                events.set_var_range(var, slider_range)

            return widget

        # Cutter select
        def draw_cutter_widget():
            var_keys = list(events.cutter_vars.keys())
            widget_dict = {var: make_cutter_widget(var) for var in var_keys}

            cutter_select = widgets.Select(
                options=var_keys,
                value=var_keys[0] if var_keys else None,
                description="",
                layout=widgets.Layout(width="200px", height="150px"),
            )

            # Shows the slider/radio button for whichever variable is selected
            active_box = widgets.VBox(
                [widget_dict[var_keys[0]]] if var_keys else [],
                layout=widgets.Layout(min_height="60px"),
            )

            def on_select_change(change):
                selected = change["new"]
                if selected and selected in widget_dict:
                    active_box.children = [widget_dict[selected]]
                    draw_cutter_plot(selected)

            cutter_select.observe(on_select_change, names="value")

            # Pre-cache arrays and draw initial cutter plot
            for var in var_keys:
                events.get_array(var)
            if var_keys:
                draw_cutter_plot(var_keys[0])

            return (cutter_select, widget_dict, active_box)

        def update_cutter_widget(var: str, cutter_tuple):
            cutter_select, widget_dict, active_box = cutter_tuple

            if var not in widget_dict:
                widget_dict[var] = make_cutter_widget(var)
                cutter_select.options = list(widget_dict.keys())

            cutter_select.value = var
        
        def button_func(button):
            events.export_vars()

        # Wire up observers
        cutter_tuple = draw_cutter_widget()
        cutter_select, _, active_box = cutter_tuple

        display_variable_dropdown.observe(lambda _: draw_base_plot(), names="value")
        bins_slider.observe(lambda _: draw_base_plot(), names="value")
        cutter_variable_dropdown.observe(
            lambda change: update_cutter_widget(change["new"], cutter_tuple),
            names="value",
        )

        export_button.on_click(button_func)

        draw_base_plot()

        # Layout
        left_panel  = widgets.VBox([display_variable_dropdown, bins_slider, base_fig])
        right_panel = widgets.VBox([cutter_variable_dropdown, cutter_select, active_box, export_button, cutter_fig])

        display(widgets.HBox([left_panel, right_panel]))

## Default cutter variables
Change these to adjust which cutter variables show up by default for each particle type. If you're unsure about whether to include a cutter variable, you can select and cut with any variable in the cutting block instead

In [18]:
muon_cutter_vars = ["B0_MCORR",
                    "B0_PT",
                    "B0_ENDVERTEX_CHI2",
                    "B0_IPCHI2_OWNPV",
                    "Kplus_ProbNNk",
                    "piminus_ProbNNk",
                    "Kplus_ProbNNmu",
                    "Kplus_ProbNNe",
                    "muplus_ProbNNe",
                    "muminus_ProbNNe",
                    "Kplus_ProbNNp",
                    "muplus_ProbNNp",
                    "muminus_ProbNNp",
                    "Kplus_ProbNNpi",
                    "muplus_ProbNNpi",
                    "muminus_ProbNNpi",
                    "Kplus_isMuon",
                    "piminus_isMuon"

]
electron_cutter_vars = ["B0_MCORR",
                    "B0_PT",
                    "B0_ENDVERTEX_CHI2",
                    "B0_IPCHI2_OWNPV",
                    "Kplus_ProbNNk",
                    "Kplus_ProbNNmu",
                    "Kplus_ProbNNe",
                    "eplus_ProbNNe",
                    "eminus_ProbNNe",
                    "Kplus_ProbNNp",
                    "eplus_ProbNNp",
                    "eminus_ProbNNp",
                    "Kplus_ProbNNpi",
                    "eplus_ProbNNpi",
                    "eminus_ProbNNpi",
]

## File selection
Run this block to select the root file you'd like to use for cutting.
You can change your file selection without needing to re-run this block.

In [19]:
fc = FileChooser("data/")
fc.filter_pattern = "*.root"

display(fc)

FileChooser(path='/Users/elliotlyons/Desktop/Classwork/CERN Project PRO1000/Project-116/data', filename='', ti…

## Cutting program
Run this block to choose and cut variables dynamically using a UI. Currently, we're looking for a peak in B0_MM at ~5,279.5 MeV. The range sliders and plots cut out extreme outliers, which means that the data will always be slightly mutated (typically a ~3% loss) by default.

In [50]:
# Create an Events object with our selected root file
events = Events(fc)

# Set default cutter variables using lists in a previous block
events.cutter_vars = events.determine_default_vars({"muon": muon_cutter_vars, "electron": electron_cutter_vars})

# Initialize and display our UI
ui_helper = UIHelper(events)
ui_helper.display_ui()


00385270_00000001_1.dvntuple.root contains tree mydecaytree_muons;1 of type muon


TypeError: numpy boolean subtract, the `-` operator, is not supported, use the bitwise_xor, the `^` operator, or the logical_xor function instead.

TypeError: numpy boolean subtract, the `-` operator, is not supported, use the bitwise_xor, the `^` operator, or the logical_xor function instead.